In [ ]:
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load data
X_train = pd.read_csv("../data/train_test/X_train_smote.csv")
X_test = pd.read_csv("../data/train_test/X_test.csv")
y_test = pd.read_csv("../data/train_test/y_test.csv").squeeze()

# Load model + scaler
model = joblib.load("../models/best_model.pkl")
scaler = joblib.load("../models/scaler.pkl")

# Get expected feature names
expected_cols = scaler.feature_names_in_

# Align columns
X_train = X_train.reindex(columns=expected_cols, fill_value=0)
X_test = X_test.reindex(columns=expected_cols, fill_value=0)

# Scale
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Keep feature names
X_test_scaled = pd.DataFrame(X_test_scaled, columns=expected_cols)

# Prediction
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))


ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- AccountStatus_Closed
- AccountStatus_Pending
- AccountStatus_Suspended
- AgeCategory_25-34
- AgeCategory_35-44
- ...


In [21]:
import pandas as pd
import joblib
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# =========================
# Chargement des données
# =========================
X_test = pd.read_csv("../data/train_test/X_test.csv")
y_test = pd.read_csv("../data/train_test/y_test.csv").squeeze()

# =========================
# Chargement modèle + scaler
# =========================
scaler = joblib.load("../models/scaler.pkl")
model = joblib.load("../models/best_model.pkl")

# =========================
# Colonnes numériques
# =========================
num_cols = scaler.feature_names_in_

# =========================
# Séparation features
# =========================
X_test_num = X_test[num_cols]
X_test_other = X_test.drop(columns=num_cols, errors='ignore')

# =========================
# Scaling
# =========================
X_test_scaled_num = scaler.transform(X_test_num)

X_test_scaled_num = pd.DataFrame(
    X_test_scaled_num,
    columns=num_cols,
    index=X_test.index
)

# =========================
# Recomposition
# =========================
X_test_final = pd.concat([X_test_scaled_num, X_test_other], axis=1)

# Alignement avec le modèle
X_test_final = X_test_final.reindex(columns=model.feature_names_in_, fill_value=0)

print("Columns aligned:", X_test_final.columns.equals(pd.Index(model.feature_names_in_)))

# =========================
# Probabilités
# =========================
y_proba = model.predict_proba(X_test_final)[:, 1]

# =========================
# Threshold tuning
# =========================
thresholds = [0.2, 0.25, 0.3, 0.35, 0.4]

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    
    print(f"\nThreshold: {t}")
    print("Recall:", recall_score(y_test, y_pred_t))
    print("Precision:", precision_score(y_test, y_pred_t))
    print("F1:", f1_score(y_test, y_pred_t))

# =========================
# Choix d'un threshold final (ex: 0.2)
# =========================
y_pred = (y_proba >= 0.2).astype(int)

# =========================
# Évaluation finale
# =========================
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))

# Distribution
print("\nPredictions distribution:")
print(pd.Series(y_pred).value_counts())

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Probabilities range
print("\nMin proba:", y_proba.min())
print("Max proba:", y_proba.max())


Columns aligned: True

Threshold: 0.2
Recall: 0.5773195876288659
Precision: 0.9824561403508771
F1: 0.7272727272727273

Threshold: 0.25
Recall: 0.49140893470790376
Precision: 1.0
F1: 0.6589861751152074

Threshold: 0.3
Recall: 0.10996563573883161
Precision: 1.0
F1: 0.19814241486068113

Threshold: 0.35
Recall: 0.0
Precision: 0.0
F1: 0.0

Threshold: 0.4
Recall: 0.0
Precision: 0.0
F1: 0.0

Accuracy: 0.856
Precision: 0.9824561403508771
Recall: 0.5773195876288659
F1-score: 0.7272727272727273

Predictions distribution:
0    704
1    171
Name: count, dtype: int64

Confusion Matrix:
[[581   3]
 [123 168]]

Min proba: 0.07
Max proba: 0.33


c:\Users\chaie\OneDrive\Desktop\projetML\projetML\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\chaie\OneDrive\Desktop\projetML\projetML\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
